In [14]:
# [escape mode]에서: 
# A(above) / B(below) : create new cell
# DD : delete a cell
# M : code mode >> markdown mode
# Y : markdown mode >> code mode

# [enter] : [escape mode] to [edit mode] 

# [edit mode]에서:
# [shift] + [enter] : run current cell & create new cell
# [ctrl] + [enter] : run current cell only

# 3.0 LLMs and Chat models

In [2]:
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI, ChatAnthropic
import os
from dotenv import dotenv_values

env_vars = dotenv_values('.env')

os.environ['OPENAI_API_KEY'] = env_vars.get('OPENAI_API_KEY')

llm = OpenAI()
chat = ChatOpenAI()

# a = llm.predict("How many planets are there?")  ### text-davinci-003 >>> NOT GONNA USE IT! (outdated & expensive)
b = chat.predict("How many planets are there?")  ### gpt-3.5-turbo >>> Newer, Cheaper, and based on davinci-003

# a, b

# 3.1 Predict Messages



* AIMessage : AI가 생성하는 메세지  
* SystemMessage : LLM에 설정을 하기 위한 메세지

In [3]:

from langchain.schema import HumanMessage, AIMessage, SystemMessage



chat = ChatOpenAI(temperature=0.1)

messages = [
    SystemMessage(
        content="You are a geography expert. And you only reply in Italian."
    ),
    AIMessage(content="Ciao, mi chiamo Paolo!"),
    HumanMessage(
        content="What is the distance between Mexico and Thailand? Also what is your name?"
    )
]

chat.predict_messages(messages)

AIMessage(content='La distanza tra il Messico e la Thailandia è di circa 16.000 chilometri. Come posso aiutarti oggi?')

# 3.2 Prompt Templates

* PromptTemplate : string을 이용해 template를 만듦.
* ChatPromptTemplate : template을 message로부터 만듦.

In [3]:
from langchain.schema import HumanMessage, AIMessage, SystemMessage
from langchain.prompts import PromptTemplate, ChatPromptTemplate



chat = ChatOpenAI(temperature=0.1)

template = PromptTemplate.from_template("What is the distance between {country_a} and {country_b}?")

prompt = template.format(country_a="Mexico", country_b="Thailand")

chat.predict(prompt)

'The distance between Mexico and Thailand is approximately 9,500 miles (15,300 kilometers) when measured in a straight line.'

In [4]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert. And you only reply in {language}."),
    ("ai", "Ciao, mi chiamo {name}!"),
    ("human", "What is the distance between {country_a} and {country_b}? Also what is your name?")
])

prompt2 = template.format_messages(
    language="Greek",
    name="Socrates",
    country_a="Mexico",
    country_b="Thailand"
)

chat.predict_messages(prompt2)

AIMessage(content='Γεια σας! Το όνομά μου είναι Σωκράτης. Η απόσταση μεταξύ του Μεξικού και της Ταϊλάνδης είναι περίπου 16.000 χιλιόμετρα.')

# 3.3 OutputParser and LCEL

In [5]:
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):

    def parse(self, text):
        items = text.strip().split(",")
        return list(map(str.strip, items))


p = CommaOutputParser()

p.parse("Hello, how, are, you")

['Hello', 'how', 'are', 'you']

In [43]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are a list generating machine. Everything you are asked will be \
     answered with a comma-seperated list of max {max_items} in lowercase. DO NOT reply with anything else."),
     ("human", "{question}")
])

prompt = template.format_messages(
    max_items=10, question="What are the colors?"
)

result = chat.predict_messages(prompt)

p = CommaOutputParser()

p.parse(result.content)

['red',
 'orange',
 'yellow',
 'green',
 'blue',
 'indigo',
 'violet',
 'black',
 'white',
 'gray']

In [44]:
chain = template | chat | CommaOutputParser()

chain.invoke({
    "max_items":5,
    "question": "What are the poketmons?"
})

['pikachu', 'charizard', 'bulbasaur', 'squirtle', 'jigglypuff']

# 3.4 Chaining Chains

### Components of Chain
1. Prompt
2. Retriever
3. LLM, ChatModel
4. Tool
5. OutputParser

---

#### Input Type
* Prompt > Dict
* Retriever > String
* LLM, ChatModel > String, list of chat messages or a PromptValue
* Tool > String, Dict, depending on the tool
* OutputParser > the output of an LLM or ChatModel

#### Output Type
* Prompt > PromptValue
* Retriever > List of documents
* LLM > String
* ChatModel > ChatMessage
* Tool > depending on the tool
* OutputParser > depending on the parser

In [15]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(temperature=0.1,
                  streaming=True,
                  callbacks=[StreamingStdOutCallbackHandler()])

chef_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a world-class international chef. You create easy to follow\
     recipies for any type of cuisine with easy to find ingredients.'),
    ('human', 'I want to cook {cuisine} food.')
])

chef_chain = chef_prompt | chat

veg_chef_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a vegetarian chef specialized on making traditional recipies\
     vegetarian. You find alternative ingredients and explain their preparation. You\
     don\'t radically modify the recipe. If there is no alternative for a food just\
     say you don\'t know how to replace it.'),
     ('human', '{recipe}')
])

veg_chain = veg_chef_prompt | chat

final_chain = {'recipe': chef_chain} | veg_chain

final_chain.invoke({'cuisine': 'indian'})

Great choice! Indian cuisine is known for its bold flavors and aromatic spices. Let's start with a classic Indian dish called Butter Chicken. Here's an easy recipe for you:

Ingredients:
- 500g boneless chicken, cut into bite-sized pieces
- 2 tablespoons butter
- 1 onion, finely chopped
- 2 cloves of garlic, minced
- 1-inch piece of ginger, grated
- 2 teaspoons garam masala
- 1 teaspoon turmeric powder
- 1 teaspoon chili powder (adjust according to your spice preference)
- 1 cup tomato puree
- 1/2 cup heavy cream
- Salt, to taste
- Fresh cilantro, for garnish

Instructions:
1. Heat the butter in a large pan over medium heat. Add the chopped onion and sauté until golden brown.
2. Add the minced garlic and grated ginger to the pan. Cook for another minute until fragrant.
3. In a small bowl, mix together the garam masala, turmeric powder, and chili powder. Add this spice mixture to the pan and cook for a minute to toast the spices.
4. Add the chicken pieces to the pan and cook until they 

AIMessageChunk(content="Great choice! Butter Chicken is a delicious and popular Indian dish. To make it vegetarian, we can replace the chicken with a plant-based alternative. Here's an alternative recipe for Vegetarian Butter Chicken:\n\nIngredients:\n- 500g plant-based chicken substitute (such as tofu, tempeh, or seitan), cut into bite-sized pieces\n- 2 tablespoons butter or vegan butter substitute\n- 1 onion, finely chopped\n- 2 cloves of garlic, minced\n- 1-inch piece of ginger, grated\n- 2 teaspoons garam masala\n- 1 teaspoon turmeric powder\n- 1 teaspoon chili powder (adjust according to your spice preference)\n- 1 cup tomato puree\n- 1/2 cup coconut cream or cashew cream (for a creamy texture)\n- Salt, to taste\n- Fresh cilantro, for garnish\n\nInstructions:\n1. Heat the butter in a large pan over medium heat. Add the chopped onion and sauté until golden brown.\n2. Add the minced garlic and grated ginger to the pan. Cook for another minute until fragrant.\n3. In a small bowl, mix

# 4.0 Model I/O

### Modules


<span style="background-color:yellow; color:black;"><strong> Model I/O </strong></span>

Interface with language models

* <strong>Prompts</strong>: Templatize, dynamically select, and manage model inputs
* <strong>Language models</strong>: Make calls to language models through common interfaces
* <strong>Output parsers</strong>: Extract information from model outputs


<span style="background-color:yellow; color:black;"><strong> Retrieval </strong></span>

Interface with application-specific data

How to work with external data, how to provide the data

* <strong>Document loaders</strong>
* <strong>Document transformers</strong>
* <strong>Text embedding models</strong>
* <strong>Vector stores</strong>
* <strong>Retrievers</strong>


<span style="background-color:yellow; color:black;"><strong> Chains </strong></span>

Construct sequences of calls


<span style="background-color:yellow; color:black;"><strong> Agents </strong></span>

Let chains choose which tools to use given high-level directives

The most experimental part.


<span style="background-color:yellow; color:black;"><strong> Memory </strong></span>

Persist application state between runs of a chain


<span style="background-color:yellow; color:black;"><strong> Callbacks </strong></span>

Log and stream intermediate steps of any chain

# 4.1 FewShotPromptTemplate

In [ ]:
# !conda activate chatgpt
# !pip install -r requirements.txt
# !pip install python-dotenv

In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler 
from dotenv import load_dotenv

load_dotenv()

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
    )

t = PromptTemplate.from_template("What is the capital of {country}?")
###
# t = PromptTemplate(
#     template='Wat is the capital of {country}?',
#     input_variables=['country'],
# )
###

t.format(country='France') # t == prompt template


'What is the capital of France?'

In [3]:
chat.predict('What do you know about France?')

France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. France is also known for its wine regions, fashion industry, and art scene. The country has a diverse landscape, including mountains, beaches, and countryside. French is the official language, and the currency is the Euro. France is a member of the European Union and is one of the most visited countries in the world.

'France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. France is also known for its wine regions, fashion industry, and art scene. The country has a diverse landscape, including mountains, beaches, and countryside. French is the official language, and the currency is the Euro. France is a member of the European Union and is one of the most visited countries in the world.'

In [4]:
examples = [
    {
        'question': 'What do you know about France?',
        'answer': '''
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        ''',
    },
    {
        'question': 'What do you know about Italy?',
        'answer': '''
        Here is what I know:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        ''',
    },
    {
        'question': 'What do you know about Greece?',
        'answer': '''
        Here is what I know:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        ''',
    },
]

example_template = '''
    Human: {question}
    AI: {answer}
'''

example_prompt = PromptTemplate.from_template(example_template) # param: 'Human: {question}\nAI:{answer}'로 대체 가능.

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix='Human: What do you know about {country}?',
    input_variables=['country']
)

# prompt.format(country='Germany')

chain = prompt | chat
chain.invoke({
    'country': 'Germany'
})

AI: 
        Here is what I know:
        Capital: Berlin
        Language: German
        Food: Bratwurst and Sauerkraut
        Currency: Euro

AIMessageChunk(content='AI: \n        Here is what I know:\n        Capital: Berlin\n        Language: German\n        Food: Bratwurst and Sauerkraut\n        Currency: Euro')

# 4.2 FewShotChatMessagePromptTemplate

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler 


examples = [
    {
        'country': 'France',
        'answer': '''
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        ''',
    },
    {
        'country': 'Italy',
        'answer': '''
        Here is what I know:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        ''',
    },
    {
        'country': 'Greece',
        'answer': '''
        Here is what I know:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        ''',
    },
]

example_template = '''
    Human: {question}
    AI: {answer}
'''

example_prompt = ChatPromptTemplate.from_messages([
    ('human', 'What do you know about {country}?'),
    ('ai', '{answer}')
]) # param: 'Human: {question}\nAI:{answer}'로 대체 가능.

# 예제를 형식화
example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a geography expert. You give short answers.'),
    example_prompt,
    ('human', 'What do you know about {country}?')
])

chain = final_prompt | chat
chain.invoke({
    'country': 'Germany'
})


        Here is what I know:
        Capital: Berlin
        Language: German
        Food: Bratwurst and Sauerkraut
        Currency: Euro
        

AIMessageChunk(content='\n        Here is what I know:\n        Capital: Berlin\n        Language: German\n        Food: Bratwurst and Sauerkraut\n        Currency: Euro\n        ')

# 4.3 LengthBasedExampleSelector

In [6]:

from langchain.chat_models import ChatOpenAI
from langchain.prompts import example_selector
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts.prompt import PromptTemplate
# from langchain.prompts.example_selector import LengthBasedExampleSelector
from langchain.prompts.example_selector.base import BaseExampleSelector

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)


examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]


class RandomExampleSelector(BaseExampleSelector):
    def __init__(self, examples):
        self.examples = examples

    def add_example(self, example):
        self.examples.append(example)

    def select_examples(self, input_variables):
        from random import choice

        return [choice(self.examples)]


example_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

# example_selector = LengthBasedExampleSelector(
#     examples=examples,
#     example_prompt=example_prompt,
#     max_length=150,   # few shot learning에 사용할 예제의 양
# )

example_selector = RandomExampleSelector(
    examples=examples,
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

prompt.format(country="Brazil")
# 'Human: What do you know about Italy?\nAI:\n        I know this:\n        Capital: Rome\n        Language: Italian\n        Food: Pizza and Pasta\n        Currency: Euro\n        \n\nHuman: What do you know about Brazil?'

'Human: What do you know about Greece?\nAI: \n        I know this:\n        Capital: Athens\n        Language: Greek\n        Food: Souvlaki and Feta Cheese\n        Currency: Euro\n        \n\nHuman: What do you know about Brazil?'

# 4.4 Serialization and Composition

In [25]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import load_prompt

from langchain.prompts import PromptTemplate
from langchain.prompts.pipeline import PipelinePromptTemplate



prompt = load_prompt('./prompt.yaml')

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

prompt.format(country='xxx')

intro = PromptTemplate.from_template(
    """
    You are a role playing assistant.
    And you are impersonating a {character}
"""
)

example = PromptTemplate.from_template(
    """
    This is an example of how you talk:

    Human: {example_question}
    You: {example_answer}
"""
)

start = PromptTemplate.from_template(
    """
    Start now!

    Human: {question}
    You:
"""
)

final = PromptTemplate.from_template(
    """
    {intro}
                                     
    {example}
                              
    {start}
"""
)

prompts = [
    ("intro", intro),
    ("example", example),
    ("start", start),
]


full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts,
)

# full_prompt.format(
#     character='Pirate',
#     example_question='What is your location?',
#     example_answer='Arrrg! That is a secret!! Arg Arg!!',
#     question='What is your fav food?',
# )

chain = full_prompt | chat

chain.invoke(
    {
        "character": "Pirate",
        "example_question": "What is your location?",
        "example_answer": "Arrrrg! That is a secret!! Arg arg!!",
        "question": "What is your fav food?",
    }
)
 

Arrrrg matey! Me favorite grub be a hearty plate o' salted beef and hardtack! Aye, nothing beats the taste o' the sea! Arrrrg!

AIMessageChunk(content="Arrrrg matey! Me favorite grub be a hearty plate o' salted beef and hardtack! Aye, nothing beats the taste o' the sea! Arrrrg!")

In [8]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.globals import set_llm_cache, set_debug
from langchain.cache import InMemoryCache, SQLiteCache

set_llm_cache(SQLiteCache("cache.db"))


chat = ChatOpenAI(
    temperature=0.1,
    # streaming=True,
    # callbacks=[
    #     StreamingStdOutCallbackHandler(),
    # ],
)

chat.predict("How do you make italian pasta")

'To make Italian pasta, you will need the following ingredients:\n\n- 2 cups of all-purpose flour\n- 2 large eggs\n- 1/2 teaspoon of salt\n- Water (if needed)\n\nHere is a step-by-step guide to making Italian pasta:\n\n1. On a clean work surface, pour the flour and make a well in the center.\n2. Crack the eggs into the well and add the salt.\n3. Using a fork, gradually mix the eggs into the flour until a dough starts to form.\n4. Use your hands to knead the dough until it is smooth and elastic. If the dough is too dry, add a little water. If it is too wet, add a little more flour.\n5. Wrap the dough in plastic wrap and let it rest for at least 30 minutes.\n6. After resting, roll out the dough using a pasta machine or a rolling pin until it is thin.\n7. Cut the dough into your desired shape, such as fettuccine or spaghetti.\n8. Cook the pasta in a large pot of boiling salted water for 2-3 minutes, or until al dente.\n9. Drain the pasta and toss it with your favorite sauce or toppings.\n

In [9]:
chat.predict("How do you make italian pasta")

'To make Italian pasta, you will need the following ingredients:\n\n- 2 cups of all-purpose flour\n- 2 large eggs\n- 1/2 teaspoon of salt\n- Water (if needed)\n\nHere is a step-by-step guide to making Italian pasta:\n\n1. On a clean work surface, pour the flour and make a well in the center.\n2. Crack the eggs into the well and add the salt.\n3. Using a fork, gradually mix the eggs into the flour until a dough starts to form.\n4. Use your hands to knead the dough until it is smooth and elastic. If the dough is too dry, add a little water. If it is too wet, add a little more flour.\n5. Wrap the dough in plastic wrap and let it rest for at least 30 minutes.\n6. After resting, roll out the dough using a pasta machine or a rolling pin until it is thin.\n7. Cut the dough into your desired shape, such as fettuccine or spaghetti.\n8. Cook the pasta in a large pot of boiling salted water for 2-3 minutes, or until al dente.\n9. Drain the pasta and toss it with your favorite sauce or toppings.\n

In [10]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import get_openai_callback

chat = ChatOpenAI(
    temperature=0.1,
)

with get_openai_callback() as usage:
    chat.predict('What is the recipe for soju?')
    print(usage)

Tokens Used: 273
	Prompt Tokens: 15
	Completion Tokens: 258
Successful Requests: 1
Total Cost (USD): $0.0005385


In [11]:
from langchain.chat_models import ChatOpenAI
from langchain.llms.openai import OpenAI

chat = OpenAI(
    temperature=0.1,
    max_tokens=450,
    model='gpt-3.5-turbo-16k'
)

chat.save("model.json")

In [12]:
from langchain.chat_models import ChatOpenAI
from langchain.llms.openai import OpenAI
from langchain.llms.loading import load_llm


chat = load_llm("model.json")

chat

c:\Users\hihye\OneDrive\문서\FullstackGPT\env\lib\site-packages\langchain\llms\openai.py:216: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(
c:\Users\hihye\OneDrive\문서\FullstackGPT\env\lib\site-packages\langchain\llms\openai.py:811: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(


OpenAIChat(client=<class 'openai.api_resources.chat_completion.ChatCompletion'>, model_name='gpt-3.5-turbo-16k', model_kwargs={'temperature': 0.1, 'max_tokens': 450, 'top_p': 1, 'frequency_penalty': 0, 'presence_penalty': 0, 'n': 1, 'request_timeout': None, 'logit_bias': {}})

# 5. Memory

### Common Function
* save_context()
* load_memory_variables()

### 5 types of memory
* CoversationBufferMemory : 대화 내용 전체 저장. 비효울적. 고비용. 텍스트 자동완성 기능 구현 시 유용.
* ConversationBufferWindowMemory : 대화의 특정 부분만 저장.
* ConversationSummaryMemory : 대화 내용을 요약해 저장.
* ConversationSummaryBufferMemory : buffer window memory + summary memory; limit을 정하고 그 이전 메세지들은 요약해서 기억.
* ConversationKGMemory : "Knowledge Graph"; 

In [13]:
from langchain.memory import ConversationBufferMemory


memory = ConversationBufferMemory(return_messages=True)

memory.save_context({'input': 'Hi!'}, {'output': 'How are you?'})

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi!'), AIMessage(content='How are you?')]}

In [14]:
memory.save_context({'input': 'Hi!'}, {'output': 'How are you?'})

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi!'),
  AIMessage(content='How are you?'),
  HumanMessage(content='Hi!'),
  AIMessage(content='How are you?')]}

In [15]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=4,   # 버퍼 윈도우의 사이즈, 몇 개의 메시지를 저장할지 지정 
)

def add_message(input, output):
    memory.save_context({'input': input}, {'output': output})


add_message(1, 1)
add_message(2, 2)
add_message(3, 3)
add_message(4, 4)

memory.load_memory_variables({})

{'history': [HumanMessage(content='1'),
  AIMessage(content='1'),
  HumanMessage(content='2'),
  AIMessage(content='2'),
  HumanMessage(content='3'),
  AIMessage(content='3'),
  HumanMessage(content='4'),
  AIMessage(content='4')]}

In [16]:
add_message(5, 5)

memory.load_memory_variables({})

{'history': [HumanMessage(content='2'),
  AIMessage(content='2'),
  HumanMessage(content='3'),
  AIMessage(content='3'),
  HumanMessage(content='4'),
  AIMessage(content='4'),
  HumanMessage(content='5'),
  AIMessage(content='5')]}

In [17]:
from langchain.memory import ConversationSummaryMemory
from langchain.chat_models import ChatOpenAI


llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryMemory(llm=llm)

def add_message(input, output):
    memory.save_context({'inputs': input}, {'output': output})


def get_history():
    return memory.load_memory_variables({})


add_message('Hi I\'m Nicolas, I live in South Korea', 'Wow that is so cool!')

In [18]:
add_message('South Korea is so pretty', 'I wish I could visit there')

In [19]:
get_history()

{'history': "Nicolas introduces himself as living in South Korea, and the AI responds enthusiastically, expressing a desire to visit the country because it's so pretty."}

In [20]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI


llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=50,
    return_messages=True
)

def add_message(input, output):
    memory.save_context({'inputs': input}, {'output': output})


def get_history():
    return memory.load_memory_variables({})


add_message('Hi I\'m Nicolas, I live in South Korea', 'Wow that is so cool!')
add_message('South Korea is so pretty', 'I wish I could visit there')

get_history()

{'history': [HumanMessage(content="Hi I'm Nicolas, I live in South Korea"),
  AIMessage(content='Wow that is so cool!'),
  HumanMessage(content='South Korea is so pretty'),
  AIMessage(content='I wish I could visit there')]}

In [21]:
add_message('How far is Korea from Argentina?', 'I don\'t know! Super far!')

get_history()

{'history': [SystemMessage(content="The human introduces themselves as Nicolas and mentions they live in South Korea. The AI responds with enthusiasm, saying it's cool."),
  HumanMessage(content='South Korea is so pretty'),
  AIMessage(content='I wish I could visit there'),
  HumanMessage(content='How far is Korea from Argentina?'),
  AIMessage(content="I don't know! Super far!")]}

In [22]:
from langchain.memory import ConversationKGMemory
from langchain.chat_models import ChatOpenAI


llm = ChatOpenAI(temperature=0.1)

memory = ConversationKGMemory(
    llm=llm,
    return_messages=True
)

def add_message(input, output):
    memory.save_context({'inputs': input}, {'output': output})


def get_history():
    return memory.load_memory_variables({})


add_message('Hi I\'m Nicolas, I live in South Korea', 'Wow that is so cool!')

In [23]:
memory.load_memory_variables({'inputs': 'Who is Nicolas?'})

{'history': [SystemMessage(content='On Nicolas: Nicolas lives in South Korea.')]}

In [24]:
add_message('Nicolas likes Kimchi.', 'Wow that is so cool!')
memory.load_memory_variables({'inputs': 'What does Nicolas like?'})

{'history': [SystemMessage(content='On Nicolas: Nicolas lives in South Korea. Nicolas likes Kimchi.')]}

# 5.5 Memory on LLM Chain

### LLM에 memory를 붙여서 쓰는 방법 

In [5]:

from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
)

template = """
    You are a helpful AI talking to a human.

    {chat_history}
    Human:{question}
    You:
"""

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=PromptTemplate.from_template(template),
    verbose=True,
)

chain.predict(question="My name is Nico")

chain.predict(question="I live in Seoul")



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    
    Human:My name is Nico
    You:


> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Nico
AI: Hello Nico! How can I assist you today?
    Human:I live in Seoul
    You:


> Finished chain.


"That's great to know! How can I assist you with information or tasks related to Seoul?"

In [6]:
chain.predict(question="What is my name?")



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Nico
AI: Hello Nico! How can I assist you today?
Human: I live in Seoul
AI: That's great to know! How can I assist you with information or tasks related to Seoul?
    Human:What is my name?
    You:



Retrying langchain.chat_models.openai.ChatOpenAI.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised ServiceUnavailableError: The server is overloaded or not ready yet..



> Finished chain.


'Your name is Nico.'

# 5.6 Chat Based Memory

### 

In [3]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI talking to a human"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=True,
)

chain.predict(question="My name is Nico")



> Entering new LLMChain chain...
Prompt after formatting:
System: You are a helpful AI talking to a human
Human: My name is Nico

> Finished chain.


'Nice to meet you, Nico! How can I assist you today?'

# 5.7 LCEL Based Memory

In [7]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder


llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI talking to a human"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)


def load_memory(_):
    return memory.load_memory_variables({})["history"]


chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm


def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    print(result)

In [8]:

invoke_chain("My name is nico")

invoke_chain("What is my name?")

content='Nice to meet you, Nico! How can I assist you today?'
content='Your name is Nico.'


# 6.0 RAG (Retrieval Augumented Generation)

private document를 question과 함께 context 형태로 전달해 해당 document에 대한 내용 추가 학습.
-> tuning과 비슷한 개념?


### Retrieve
source >> load >> transform(split data) >> embed >> store >> retrieve(search) 



# 6.1 Data Loaders and Splitters

In [17]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader, TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators='\n',
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
)

# loader = TextLoader("./files/free_text.txt")
loader = UnstructuredFileLoader("./files/chapter_one.pdf")

len(loader.load_and_split(text_splitter=splitter))

16

# 6.2 Tiktoken

OpenAI에 의해 만들어진 tokenizer의 일종.

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)


loader = UnstructuredFileLoader("./files/chapter_one.docx")

# 6.3 Vectors
<pre>
   (e.g.) <3-dimension vectors>
   : 실제 모델은 1000개 이상의 차원으로 이루어진 벡터 사용.
     수치화에 따른 단어 사이 관계성 추정.

            M     |    F    |    R    
   King    0.9    |   0.1   |   1.0
   Man     0.9    |   0.1   |    0
   ==================================
   (King - Man)

   <span style="background-color:blue;">Royal</span>    0     |    0    |    1.0
   </pre>

In [24]:
from langchain.embeddings import OpenAIEmbeddings


embedder = OpenAIEmbeddings()
embedder.embed_query("Hi")

vector = embedder.embed_documents([
    "Hi",
    "How are you?",
    "longer sentences is allowed because the subject of embedding is a document!"
])

print(len(vector), len(vector[0]))

3 1536


In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import Chroma
from langchain.storage import LocalFileStore

cache_dir = LocalFileStore("./.cache/")


splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
loader = UnstructuredFileLoader("./files/chapter_one.pdf")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = Chroma.from_documents(docs, cached_embeddings)

In [22]:
results = vectorstore.similarity_search("where does winston live")

results

Number of requested results 4 is greater than number of elements in index 3, updating n_results = 3


[Document(page_content="It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions (…).\n5\n10\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine (…), went slowly, resting several times on the way. On each landing, the poster with the enormous face gazed from the wall. It

# 6.6 RetrievalQA

### [Legacy] LLMChain
> Recommend using LCEL(LangChain Expression Language)

* Retriever : Interface which makes document retrieve. 

* Chain Types
- Stuff documents chain : 모든 docs를 하나의 context로 만들어 전달.
- Refine documents chain : 여러 개의 docs를 반복하면서 질문에 대한 답을 업데이트하는 방식. 비쌈.
- Map reduce documents chain : 여러 개의 docs에 대해 각각 요약본을 만들어 전달.
- Map re-rank documents chain : 여러 개의 docs에 대해 각각 답변을 생성하고 점수를 매겨 best answer를 반환.

In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.chains import RetrievalQA
from dotenv import dotenv_values
import os

env_vars = dotenv_values('.env')
os.environ['OPENAI_API_KEY'] = env_vars.get('OPENAI_API_KEY')

llm = ChatOpenAI()

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
loader = UnstructuredFileLoader("./files/free_text.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
)

# chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="map_rerank",
#     retriever=vectorstore.as_retriever(),
# )

chain.run("Describe Victory Mansions")

c:\Users\hihye\anaconda3\lib\site-packages\langchain\chains\llm.py:349: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


"I don't know"

# 6.8 Stuff LCEL Chain

In [8]:

from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    temperature=0.1,
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
loader = UnstructuredFileLoader("./files/free_text.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriver = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}",
        ),
        ("human", "{question}"),
    ]
)

chain = (
    {
        "context": retriver,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

chain.invoke("Describe Victory Mansions")

AIMessage(content="I don't have information about Victory Mansions in the provided text.")

# 6.9 Map Reduce LCEL Chain

In [ ]:

from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
import os
from dotenv import dotenv_values

env_vars = dotenv_values('.env')
os.environ['OPENAI_API_KEY'] = env_vars.get('OPENAI_API_KEY')
os.environ['LANGSMITH_API_KEY'] = env_vars.get('LANGSMITH_API_KEY')

llm = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
loader = UnstructuredFileLoader("./files/free_text.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriver = vectorstore.as_retriever()

map_doc_prompt = ChatPromptTemplate.format_messages(
    [
        (
            "system",
            """
                Use the following portion of a long document to see if any of the text is
                relevant to answer the question. Return any relevant text verbatim.
                -----
                {context}
            """
        ),
        ("human", "{question}")
    ]
)

map_doc_chain = map_doc_prompt | llm

def map_docs(inputs):
    documents = inputs["documents"]
    question = inputs["question"]

    results = []
    # for doc in documents:
    #     result = map_doc_chain.invoke({
    #                 "context": doc.page_content,
    #                 "question": question
    #             }).content
    #     results.append(result)
    # # print(results)
    # results = "\n\n".join(results)
    # return results

    return "\n\n".join(map_doc_chain.invoke({
                                                "context": doc.page_content,
                                                "question": question
                                            }).content for doc in documents)


map_chain = { "documents": retriver, "question": RunnablePassthrough() } | RunnableLambda(map_docs)

# list of docs
# for doc in docs | prompt | llm
# final doc | prompt | llm

final_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """
        Given the following extracted parts of a long document and a
        question, create a final answer.
        If you don't know the answer, just say that you don't know.
        Don't try to make up an answer.
        -----
        {context}
     """
    ),
    ("human", "{question}")
])

chain = {
            "context": map_chain,
            "question": RunnablePassthrough()
        } | final_prompt | llm

chain.invoke("Describe Victory Mansions")

# 8. Private GPT

## 8.1 HuggingFace Model 
* Using API by <strong>Inference Providers</strong>
* For free
* Doesn't garantee the privacy of data. ( Data would send to HuggingFace )


> Model : Mistral-7B-v0.1 (Text Generation Model) from HuggingFace

In [ ]:
from langchain.llms import HuggingFaceHub
from langchain.prompts import PromptTemplate


prompt = PromptTemplate.from_template("[INST] What is the meaning of {word} [/INST]")


llm = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    model_kwargs={
        "max_new_tokens": 512,
    }
)

chain = prompt | llm

chain.invoke({
    "word": "deploy"
})

c:\Users\hihye\OneDrive\문서\FullstackGPT\env\lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


'[INST] What is the meaning of deploy [/INST]In the context of software development, "deploy" refers to the process of making a software application or system available to users, typically by uploading it to a server or cloud-based infrastructure. This can include installing the software, configuring it, and ensuring it is running correctly. The goal of deployment is to make the software accessible and usable for its intended audience.\n\nDeployment can also refer to the act of making something operational or available in other contexts, such as in the military or business. For example, deploying troops to a foreign country or deploying a new product in a retail store.\n\nIn summary, deploy means to put something into operation or make it available for use.'

## 8.2 HuggingFacePipeline
* Using HuggingFace Model <strong>by downloading the model.</strong>
* Need time to download and a room for the model on your device to use the model.

In [1]:
from langchain.llms.huggingface_pipeline import HuggingFacePipeline
from langchain.prompts import PromptTemplate


prompt = PromptTemplate.from_template("A {word} is a")

llm = HuggingFacePipeline.from_model_id(
    model_id="gpt2",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 50
    }
)

chain = prompt | llm

chain.invoke({
    "word": "deploy"
})

c:\Users\hihye\OneDrive\문서\FullstackGPT\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hihye\OneDrive\문서\FullstackGPT\env\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\hihye\OneDrive\문서\FullstackGPT\env\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hihye\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disable

'A deploy is a continuous deployment, providing a continuous flow of data from different nodes over time. The deployment can be one-off, or continuous with other nodes that need to be deployed sequentially due to resource exhaustion. Deployment occurs in two ways. First, deploying'

## 8.3 Downloaded Model
* From <strong>GPT4All</strong> (application)
* Need a room for downloading models on user's local device.

In [ ]:
from langchain.llms import GPT4All
from langchain.prompts import PromptTemplate


prompt = PromptTemplate.from_template(
    "You are a helpful assistant that defines words. Define this word: {word}."
)

llm = GPT4All(
    model="./falcon.bin",
)

chain = prompt | llm

chain.invoke({
    "word": "deploy"
})

## 8.4 Ollama
* Need a space for downloading models on your own device.
* Complimentary & garantee users' privacy.
* Code : 02_PrivateGPT.py

# 9. Quiz GPT
## 9.8 Function Calling

In [ ]:
from langchain_community.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate


def get_weather(lon, lat):
    print("call an API...")

schema = {
    "name": "get_weather",
    "description": "function that takes longitude and latitude \
        to find the weather of a place",
    "parameters": {
        "type": "object",
        "properties": {
            "lon": {
                "type": "string",
                "description": "The longitude coordinate"
            },
            "lat": {
                "type": "string",
                "description": "The latitude coordinate"
            },
        },
    },
    "required": ["lon", "lat"],
}

llm = ChatOpenAI(
    temperature=0.1,
).bind(
    function_call="auto",
    functions=[
        schema
    ]
)

prompt = PromptTemplate.from_template("How is the weather in {city}?")

chain = prompt | llm

response = chain.invoke({"city": "rome"})
res_args = response.additional_kwargs["function_call"]["arguments"]

print(res_args)

{"lon":"12.4964","lat":"41.9028"}


In [19]:
import json

r = json.loads(res_args)

print(r)
# get_weather(r['lon'], r['lat'])

{'lon': '12.4964', 'lat': '41.9028'}


In [32]:
function = {
                "name": "create_quiz",
                "description": "function that takes a list of questions and answers and returns a quiz",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "questions": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "question": {
                                        "type": "string",
                                    },
                                    "answers": {
                                        "type": "array",
                                        "items": {
                                            "type": "object",
                                            "properties": {
                                                "answer": {
                                                    "type": "string",
                                                },
                                                "correct": {
                                                "type": "boolean",
                                                },
                                            },
                                            "required": ["answer", "correct"],
                                            },
                                    },
                                },
                                "required": ["question", "answers"],
                                },
                            }
                        },
                        "required": ["questions"],
                },
            }


llm = ChatOpenAI(
    temperature=0.1,
).bind(
    function_call={
        "name": "create_quiz",
    },
    functions=[
        function
    ],
)

prompt = PromptTemplate.from_template("Make a quiz about {city}")

chain = prompt | llm

In [33]:
res = chain.invoke({"city": "Rome"})
res = res.additional_kwargs["function_call"]["arguments"]

res

'{"questions":[{"question":"What is the capital city of Italy?","answers":[{"answer":"Rome","correct":true},{"answer":"Milan","correct":false},{"answer":"Florence","correct":false},{"answer":"Venice","correct":false}]},{"question":"Which ancient Roman structure is known for its gladiator contests?","answers":[{"answer":"Colosseum","correct":true},{"answer":"Pantheon","correct":false},{"answer":"Roman Forum","correct":false},{"answer":"Trevi Fountain","correct":false}]},{"question":"Who was the first Roman Emperor?","answers":[{"answer":"Julius Caesar","correct":false},{"answer":"Augustus","correct":true},{"answer":"Nero","correct":false},{"answer":"Constantine","correct":false}]},{"question":"What is the name of the river that runs through Rome?","answers":[{"answer":"Tiber","correct":true},{"answer":"Arno","correct":false},{"answer":"Po","correct":false},{"answer":"Adige","correct":false}]}]}'

# 11. Meeting GPT
## 11.1 Audio Extraction

In [31]:
import subprocess

def extract_audio_from_video(video_path, audio_path):
    command = ["ffmpeg", "-y", "-i", video_path, "-vn", audio_path]

    subprocess.run(command, shell=True)


extract_audio_from_video(r"./files/podcast.mp4", r"./files/podcast-test.mp3")

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

## 11.2 Cutting the audio

In [ ]:
from pydub import AudioSegment

track = AudioSegment.from_mp3("./files/podcast-audio.mp3")

track.duration_seconds

1033.4737414965987

In [11]:
import math

ten_minutes = 10 * 60 * 1000

chunks = math.ceil(len(track) / ten_minutes)

for i in range(chunks):
    start_time = i * ten_minutes
    end_time = (i+1) * ten_minutes
    chunk = track[start_time:end_time]
    chunk.export(f"./files/chunks/chunk_{i}.mp3", format="mp3")

Final Code

In [13]:
import subprocess
from pydub import AudioSegment
import math


def extract_audio_from_video(video_path, audio_path):
    command = ["ffmpeg", "-i", video_path, "-vn", audio_path]
    subprocess.run(command, shell=True)

def cut_audio_in_chunks(audio_path, chunk_size, chunks_folder):
    track = AudioSegment.from_mp3(audio_path)
    chunk_len = chunk_size * 60 * 1000

    chunks = math.ceil(len(track) / chunk_len)

    for i in range(chunks):
        start_time = i * chunk_len
        end_time = (i+1) * chunk_len
        chunk = track[start_time:end_time]
        chunk.export(f"{chunks_folder}/chunk_{i}.mp3", format="mp3")

In [ ]:
# Test
cut_audio_in_chunks("./files/podcast-audio.mp3", 10, "./files/chunks")

## 11.3 Whisper Transcription

In [ ]:
import openai
import os
from dotenv import dotenv_values
import glob

env_vars = dotenv_values('.env')

os.environ['OPENAI_API_KEY'] = env_vars.get('OPENAI_API_KEY')

transcript = openai.audio.transcriptions.create(
    model="whisper-1", 
    file=open("./files/chunks/chunk_0.mp3", "rb"),
)

transcript.text

### Guide for Updated Model: https://platform.openai.com/docs/guides/speech-to-text

"Hey, Lindsay, how are you? Hey, Aubrey, doing great. How's it going? Excellent. I'm curious. What verb tense do you use when telling a story? I feel like I use the present tense a lot. Yeah. A lot. Just the present tense? But not just the present tense, though. Not just that. I mean, that would be kind of weird. I think I move between the present tense, the past tense, the present perfect tense, past perfect tense. Totally. Yeah. We use them all, right? It depends on what we're saying. I agree. Present tense comes up a lot because you're trying to pull someone into the story, make them feel like they were there with you. So we move into the present tense. But we use subjunctive, conditional. We use all of the tenses when telling a story. So you can't just think, this happened in the past, I'm using past simple tense. It doesn't work that way. Yeah. We just move seamlessly in and out of them, right? That's the key. We did do an episode recently on reporting our speech and our thoughts,

In [ ]:
import glob

def transcribe_chunks(chunk_folder, destination):
    files = glob.glob(f"{chunk_folder}/*.mp3")

    for file in files:
        with open(file, "rb") as audio_file, open(destination, "a") as text_file:
            transcript = openai.audio.transcriptions.create(
                                model="whisper-1", 
                                file=audio_file,
                        )
            text_file.write(transcript.text)

transcribe_chunks("./files/chunks", "./files/transcript.txt")

# Investor GPT

## 12.1 Your First Agent

In [1]:
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.1,
)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

llm.invoke(prompt)

/tmp/ipykernel_8137/3875526001.py:3: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


AIMessage(content='= $4923.38', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 59, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--85a687aa-ece3-4c58-bffa-af7b7c929806-0')

## 12.2 Structured Chat Zero-shot ReAct Agent

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.tools import StructuredTool
from langchain.agents import initialize_agent, AgentType

def plus(a, b):
    return a + b


llm = ChatOpenAI(
    temperature=0.1,
)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    tools=[
        StructuredTool.from_function(
            func=plus,
            name="Sum Calculator",
            description="Use this to perform sums of two numbers. This tool take \
            two arguments, both should be numbers."
        )
    ])

agent.invoke(prompt)



> Entering new AgentExecutor chain...
Thought: I can use the Sum Calculator tool to find the total cost of all these amounts added together.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {
    "a": 355.39,
    "b": 924.87
  }
}
```
Observation: 1280.26
Thought:I made a mistake by only summing the first two numbers. Let's continue adding the remaining numbers to get the total cost.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {
    "a": 1280.26,
    "b": 721.2
  }
}
```
Observation: 2001.46
Thought:I need to keep adding the remaining numbers to find the total cost accurately.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {
    "a": 2001.46,
    "b": 1940.29
  }
}
```
Observation: 3941.75
Thought:I have calculated the sum of all the amounts correctly up to this point. Let's continue adding the remaining numbers to find the total cost accurately.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {
    "a": 3941.75,
    "b": 57

{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost is $5273.38'}

## 12.3 Zero-shot ReAct Agent

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType

def plus(input):
    a, b = input.split(",")
    return float(a) + float(b)


llm = ChatOpenAI(
    temperature=0.1,
)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    tools=[
        Tool.from_function(
            func=plus,
            name="Sum Calculator",
            description="Use this to perform sums of two numbers. Use this tool\
                by sending a pair of number separated by a comma.\nExample: 1,2"
        )
    ])

agent.invoke(prompt)



> Entering new AgentExecutor chain...
I need to add up all these numbers to find the total cost.
Action: Sum Calculator
Action Input: 355.39, 924.87
Observation: 355.39 924.87
Thought:I need to continue adding the remaining numbers to the sum.
Action: Sum Calculator
Action Input: 1280.26, 721.2
Observation: 1280.26 721.2
Thought:I need to continue adding the remaining numbers to the sum.
Action: Sum Calculator
Action Input: 2001.46, 1940.29
Observation: 2001.46 1940.29
Thought:I need to continue adding the remaining numbers to the sum.
Action: Sum Calculator
Action Input: 3941.75, 573.63
Observation: 3941.75 573.63
Thought:I need to continue adding the remaining numbers to the sum.
Action: Sum Calculator
Action Input: 4515.38, 65.72
Observation: 4515.38 65.72
Thought:I need to continue adding the remaining numbers to the sum.
Action: Sum Calculator
Action Input: 4581.1, 35.00
Observation: 4581.1 35.00
Thought:I need to continue adding the remaining numbers to the sum.
Action: Sum Cal

{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost is $5273.38'}

## 12.4 OpenAI Functions Agent

In [15]:
from langchain.chat_models import ChatOpenAI
from typing import Type
from langchain.tools import Tool, BaseTool
from pydantic import BaseModel, Field
from langchain.agents import initialize_agent, AgentType

def plus(a, b):
    return a + b

llm = ChatOpenAI(
    temperature=0.1,
)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

class CalculatorToolArgsSchema(BaseModel):
    a:float = Field(description="The first number")
    b:float = Field(description="The second number")

class CalculatorTool(BaseTool):
    name:str = "CalculatorTool"
    description:str = """
        Use this to perform sums of two numbers.
        The first and second arguments should be numbers.
        Only receives two arguments.
        """
    args_schema : Type[CalculatorToolArgsSchema] = CalculatorToolArgsSchema

    def _run(self, a, b):
        return a + b


agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        CalculatorTool()
    ])

agent.invoke(prompt)



> Entering new AgentExecutor chain...

Invoking: `CalculatorTool` with `{'a': 355.39, 'b': 924.87}`


1280.26
Invoking: `CalculatorTool` with `{'a': 1280.26, 'b': 721.2}`


2001.46
Invoking: `CalculatorTool` with `{'a': 2001.46, 'b': 1940.29}`


3941.75
Invoking: `CalculatorTool` with `{'a': 3941.75, 'b': 573.63}`


4515.38
Invoking: `CalculatorTool` with `{'a': 4515.38, 'b': 65.72}`


4581.1
Invoking: `CalculatorTool` with `{'a': 4581.1, 'b': 35}`


4616.1
Invoking: `CalculatorTool` with `{'a': 4616.1, 'b': 552}`


5168.1
Invoking: `CalculatorTool` with `{'a': 5168.1, 'b': 76.16}`


5244.26
Invoking: `CalculatorTool` with `{'a': 5244.26, 'b': 29.12}`


5273.38The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12 is $5273.38.

> Finished chain.


{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12 is $5273.38.'}

## 12.5 Search Tool

In [ ]:
from langchain.chat_models import ChatOpenAI
from typing import Type
from langchain.tools import Tool, BaseTool
from pydantic import BaseModel, Field
from langchain.agents import initialize_agent, AgentType
from langchain.tools import DuckDuckGoSearchResults


def plus(a, b):
    return a + b

llm = ChatOpenAI(
    temperature=0.1,
)

class StockMarketSymbolSearchToolArgsSchema(BaseModel):
    query:str = Field(description="The query you will search for")

class StockMarketSymbolSearchTool(BaseTool):
    name:str = "StockMarketSymbolSearchTool"
    description:str = """
        Use this tool to find the stock market symbol for a company.
        It takes a query as an argument.
        Example query: Stock Market Symbol for Apple Company
    """

    args_schema: Type[StockMarketSymbolSearchToolArgsSchema] = StockMarketSymbolSearchToolArgsSchema

    def _run(self, query):
        ddg = DuckDuckGoSearchResults()
        return ddg.run(query)


agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        StockMarketSymbolSearchTool()
    ])

prompt = "Give em information on Cloudflare's stock and help me analyze if it's a \
    potential good investment. Also tell me what symbol does the stock have."

agent.invoke(prompt)

/tmp/ipykernel_751388/3153416365.py:11: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(
/tmp/ipykernel_751388/3153416365.py:33: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built



> Entering new AgentExecutor chain...

Invoking: `StockMarketSymbolSearchTool` with `{'query': 'Cloudflare stock'}`


I have found the stock market symbol for Cloudflare. It is "NET".

Now, let's analyze Cloudflare's stock to see if it's a potential good investment. To do this, we can look at factors such as the company's financial performance, growth potential, market trends, and analyst recommendations. Let's start by looking at Cloudflare's recent stock performance and key financial metrics.
Cloudflare's stock symbol is "NET".

Now, let's analyze Cloudflare's stock to see if it's a potential good investment. To do this, we can look at factors such as the company's financial performance, growth potential, market trends, and analyst recommendations. Let's start by looking at Cloudflare's recent stock performance and key financial metrics.

> Finished chain.


{'input': "Give em information on Cloudflare's stock and help me analyze if it's a     potential good investment. Also tell me what symbol does the stock have.",
 'output': 'I have found the stock market symbol for Cloudflare. It is "NET".\n\nNow, let\'s analyze Cloudflare\'s stock to see if it\'s a potential good investment. To do this, we can look at factors such as the company\'s financial performance, growth potential, market trends, and analyst recommendations. Let\'s start by looking at Cloudflare\'s recent stock performance and key financial metrics.\nCloudflare\'s stock symbol is "NET".\n\nNow, let\'s analyze Cloudflare\'s stock to see if it\'s a potential good investment. To do this, we can look at factors such as the company\'s financial performance, growth potential, market trends, and analyst recommendations. Let\'s start by looking at Cloudflare\'s recent stock performance and key financial metrics.'}

## 12.6 Stock Information Tools

In [ ]:
from langchain.chat_models import ChatOpenAI
from typing import Type
from langchain.tools import Tool, BaseTool
from pydantic import BaseModel, Field
from langchain.agents import initialize_agent, AgentType
from langchain.tools import DuckDuckGoSearchResults
import os
import requests


llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4o-mini"
)



class StockMarketSymbolSearchToolArgsSchema(BaseModel):
    query:str = Field(description="The query you will search for")

class StockMarketSymbolSearchTool(BaseTool):
    name:str = "StockMarketSymbolSearchTool"
    description:str = """
        Use this tool to find the stock market symbol for a company.
        It takes a query as an argument.
        Example query: Stock Market Symbol for Apple Company
    """

    args_schema: Type[StockMarketSymbolSearchToolArgsSchema] = StockMarketSymbolSearchToolArgsSchema

    def _run(self, query):
        ddg = DuckDuckGoSearchResults()
        return ddg.run(query)

class CompanyOverviewToolArgsSchema(BaseModel):
    symbol:str = Field(description="Stock Symbol of the company.\
                       Example: AAPL, TSLA")

class CompanyOverviewTool(BaseTool):
    name:str = "CompanyOverview"
    description:str = """
    Use this to get an overview of the financials of the company.
    You should enter a stock symbol.
    """
    args_schema: type[CompanyOverviewToolArgsSchema] = CompanyOverviewToolArgsSchema

    def _run(self, symbol):
        r = requests.get(f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_key}")
        return r.json()
    
class CompanyIncomeStatementTool(BaseTool):
    name:str = "CompanyIncomeStatement"
    description:str = """
    Use this to get the income statement of the financials of the company.
    You should enter a stock symbol.
    """
    args_schema: type[CompanyOverviewToolArgsSchema] = CompanyOverviewToolArgsSchema

    def _run(self, symbol):
        r = requests.get(f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_vantage_api_key}")
        return r.json()['annualReports']

class CompanyStockPerformanceTool(BaseTool):
    name:str = "CompanyStockPerformance"
    description:str = """
    Use this to get the weekly performance of the financials of the company.
    You should enter a stock symbol.
    """
    args_schema: type[CompanyOverviewToolArgsSchema] = CompanyOverviewToolArgsSchema

    def _run(self, symbol):
        r = requests.get(f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&apikey={alpha_vantage_api_key}")
        return r.json()
    

agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        StockMarketSymbolSearchTool(),
        CompanyOverviewTool(),
        CompanyIncomeStatementTool(),
        CompanyStockPerformanceTool(),
    ])

prompt = "Give me information on Cloudflare's stock, considering its financials, income statements, \
    and weekly stock performances. Help me analyze if it's a potential good investment."

agent.invoke(prompt)



> Entering new AgentExecutor chain...

Invoking: `StockMarketSymbolSearchTool` with `{'query': 'Cloudflare'}`



Invoking: `CompanyOverview` with `{'symbol': 'NET'}`


{'Symbol': 'NET', 'AssetType': 'Common Stock', 'Name': 'Cloudflare Inc', 'Description': 'CloudFlare, Inc. operates a cloud platform that offers a range of network services to companies around the world. The company is headquartered in San Francisco, California.', 'CIK': '1477333', 'Exchange': 'NYSE', 'Currency': 'USD', 'Country': 'USA', 'Sector': 'TECHNOLOGY', 'Industry': 'SERVICES-PREPACKAGED SOFTWARE', 'Address': '101 TOWNSEND ST., SAN FRANCISCO, CA, US', 'OfficialSite': 'https://www.cloudflare.com', 'FiscalYearEnd': 'December', 'LatestQuarter': '2025-03-31', 'MarketCapitalization': '45862162000', 'EBITDA': '-36037000', 'PERatio': 'None', 'PEGRatio': '2.355', 'BookValue': '3.034', 'DividendPerShare': 'None', 'DividendYield': 'None', 'EPS': '-0.23', 'RevenuePerShareTTM': '4.89', 'ProfitMargin': '-0.0472', 'OperatingMa

{'input': "Give me information on Cloudflare's stock, considering its financials, income statements,     and weekly stock performances. Help me analyze if it's a potential good investment.",
 'output': "### Cloudflare Inc. (Ticker: NET) Overview\n\n- **Company Description**: Cloudflare, Inc. operates a cloud platform that offers a range of network services to companies around the world. The company is headquartered in San Francisco, California.\n- **Market Capitalization**: $45.86 billion\n- **Sector**: Technology\n- **Industry**: Services - Prepackaged Software\n- **Official Website**: [Cloudflare](https://www.cloudflare.com)\n\n### Financial Overview\n\n- **Latest Quarter**: March 31, 2025\n- **Total Revenue (TTM)**: $1.67 billion\n- **Gross Profit (TTM)**: $1.29 billion\n- **Operating Income**: -$154.76 million\n- **Net Income**: -$78.8 million\n- **EBITDA**: -$36.04 million\n- **Earnings Per Share (EPS)**: -$0.23\n- **Profit Margin**: -4.72%\n- **Operating Margin**: -7.4%\n- **Quar

## 12.8 SQLDatabase Toolkit

In [14]:
from langchain.agents import create_sql_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.sql_database import SQLDatabase


llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4o-mini"
)

db = SQLDatabase.from_uri("sqlite:///movies.sqlite")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

agent.invoke("Give me the 5 directors that have the highest grossing films.")





> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables  
Action Input: ""  I need to check the available tables in the database to find the relevant ones for directors and films. 

Action: sql_db_list_tables  
Action Input: ""  I need to check the available tables in the database to find the relevant ones for directors and films. 

Action: sql_db_list_tables  
Action Input: ""  

ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `I don't know.`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 

In [ ]:
agent.invoke(
    "Give me the movies that have the highest votes but the lowest budgets \
             and give me the name of their directors also include their gross revenue."
)



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables  
Action Input: ""  I need to check the tables available in the database to find relevant information about movies, votes, budgets, and directors. 

Action: sql_db_list_tables  
Action Input: ""  I need to check the tables available in the database to find relevant information about movies, votes, budgets, and directors. 

Action: sql_db_list_tables  
Action Input: ""  

ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `I don't know.`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 